# Retail Customer Behavior Data Validation & SQL Analytics Project

## Business Problem

The retail company wants to better understand customer behavior, revenue performance, and churn risk. The goal of this analysis is to identify which customers generate the most value, which factors are associated with churn, which products and promotions perform best, and whether customer behavior can be used to predict churn risk.

The final insights can help the company improve customer retention, optimize marketing campaigns, increase revenue, and make more data-driven business decisions.

## Initial SMART Questions

1. Which customer segments generate the highest total sales and average transaction value?

2. How does churn differ by loyalty program status, income bracket, engagement level, and purchase frequency?

3. What behavioral factors are most strongly associated with churn, such as days since last purchase, website visits, app usage, support calls, and email subscriptions?

4. Which product categories, brands, and promotions contribute most to sales and customer engagement?

5. Can we build a predictive model that identifies customers with a higher likelihood of churn using customer, purchase, and engagement features?

First We inspect the data and run some housekeeping cleaning steps such as null values, formatting, schemas, and data integrity

In [113]:
import pandas as pd 
import numpy as np 

file_path = "../data/raw/retail_data.csv"

df = pd.read_csv(file_path)

df.head()




,customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,...,distance_to_store,holiday_season,season,weekend,customer_support_calls,email_subscriptions,app_usage,website_visits,social_media_engagement,days_since_last_purchase
0,1,56,Other,High,No,0,No,Divorced,3,Bachelor's,...,33.21,No,Spring,Yes,5,No,High,30,High,40
1,2,69,Female,Medium,No,2,No,Married,2,PhD,...,62.56,No,Summer,Yes,6,No,High,40,Medium,338
2,3,46,Female,Low,No,5,No,Married,3,Bachelor's,...,83.04,Yes,Winter,Yes,2,Yes,Low,89,Medium,61
3,4,32,Female,Low,No,0,No,Divorced,2,Master's,...,50.43,Yes,Winter,No,12,No,Low,12,Low,42
4,5,60,Female,Low,Yes,7,Yes,Divorced,2,Bachelor's,...,36.55,Yes,Summer,Yes,3,No,Medium,31,Low,242


In [114]:
# Check misisng values

df.isnull().sum().sort_values(ascending=False)

customer_id                 0
product_weight              0
promotion_type              0
promotion_id                0
product_shelf_life          0
                           ..
purchase_frequency          0
avg_purchase_value          0
month_of_year               0
week_of_year                0
days_since_last_purchase    0
Length: 78, dtype: int64

In [115]:
#Check  columns
df.shape
for col in df.columns:
    print(col)



customer_id
age
gender
income_bracket
loyalty_program
membership_years
churned
marital_status
number_of_children
education_level
occupation
transaction_id
transaction_date
product_id
product_category
quantity
unit_price
discount_applied
payment_method
store_location
transaction_hour
day_of_week
week_of_year
month_of_year
avg_purchase_value
purchase_frequency
last_purchase_date
avg_discount_used
preferred_store
online_purchases
in_store_purchases
avg_items_per_transaction
avg_transaction_value
total_returned_items
total_returned_value
total_sales
total_transactions
total_items_purchased
total_discounts_received
avg_spent_per_category
max_single_purchase_value
min_single_purchase_value
product_name
product_brand
product_rating
product_review_count
product_stock
product_return_rate
product_size
product_weight
product_color
product_material
product_manufacture_date
product_expiry_date
product_shelf_life
promotion_id
promotion_type
promotion_start_date
promotion_end_date
promotion_effective

In [116]:
# Check data info and types
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 78 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   customer_id                1000000 non-null  int64  
 1   age                        1000000 non-null  int64  
 2   gender                     1000000 non-null  object 
 3   income_bracket             1000000 non-null  object 
 4   loyalty_program            1000000 non-null  object 
 5   membership_years           1000000 non-null  int64  
 6   churned                    1000000 non-null  object 
 7   marital_status             1000000 non-null  object 
 8   number_of_children         1000000 non-null  int64  
 9   education_level            1000000 non-null  object 
 10  occupation                 1000000 non-null  object 
 11  transaction_id             1000000 non-null  int64  
 12  transaction_date           1000000 non-null  object 
 13  product_id   

In [117]:
#check duplicates

df.duplicated().sum()

0

In [118]:
# Check dataset size and unique counts

print("Dataset Shape:", df.shape)

print("Unique customers:", df["customer_id"].nunique())
print("Unique transactions:" , df["transaction_id"].nunique())
print("Unique products:", df["product_id"].nunique())
print("Unique Promotions:", df["promotion_id"].nunique())
print("Unique store locations:", df["store_location"].nunique())


Dataset Shape: (1000000, 78)
Unique customers: 1000000
Unique transactions: 632576
Unique products: 9999
Unique Promotions: 999
Unique store locations: 4


## Dataset Grain and Key Validation

Before performing analysis, I checked the structure of the dataset to understand what each row represents and which columns can be used as reliable identifiers. This step helps prevent incorrect joins, duplicate counting, and flawed business conclusions.

In [119]:
# Check whether Rows are unqiue transactions
print("Dupicate full rows:", df.duplicated().sum())
print("Duplicate transaction IDs:", df["transaction_id"].duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())
print("Duplicate product IDs:", df["product_id"].duplicated().sum())
print("Duplicate promotion IDs:", df["promotion_id"].duplicated().sum())


Dupicate full rows: 0
Duplicate transaction IDs: 367424
Duplicate customer IDs: 0
Duplicate product IDs: 990001
Duplicate promotion IDs: 999001


In [120]:
#Check how often transaction_id is duplicated
transaction_counts = df["transaction_id"].value_counts()
transaction_counts.describe()

transaction_counts 

transaction_id
115913    9
344167    8
620816    8
239407    8
2773      8
         ..
864593    1
21952     1
354366    1
421443    1
235348    1
Name: count, Length: 632576, dtype: int64

In [121]:
transaction_customer_check = df.groupby("transaction_id").agg({
    "customer_id": "nunique",
    "product_id": "nunique",
    "transaction_date": "nunique",
    "total_sales": "nunique"
}).sort_values(by="customer_id", ascending=False)

transaction_customer_check.head(10)

,customer_id,product_id,transaction_date,total_sales
transaction_id,,,,
115913,9,9,9,9
273197,8,8,8,8
504562,8,8,8,8
620816,8,8,8,8
239407,8,8,8,8
2773,8,8,8,8
344167,8,8,8,8
821598,7,7,7,7
240787,7,7,7,7


### Transaction ID Validation Observation

Further validation showed that repeated transaction_id values are linked to multiple unique customers, products, transaction dates, and total sales values. For example, some transaction IDs are associated with 8 or 9 different customers and transaction dates.

In clean retail data, duplicate transaction IDs can be valid if they represent multiple product line items from the same transaction for the same customer. However, in this dataset, repeated transaction IDs are tied to different customers and dates, which means transaction_id is not behaving like a true transaction-level key.

This confirms that the dataset should not be treated as a clean transaction-level dataset. Instead, customer_id remains the only reliable unique row identifier, and transaction_id should be treated as a reused or synthetic reference field.

In [122]:
#Check whether each product_id maps to one product 
product_name_check = df.groupby("product_id")["product_name"].nunique().sort_values(ascending=False)
print("Unique product names per product ID:")
print(product_name_check.head(10))

Unique product names per product ID:
product_id
1       4
6670    4
6663    4
6664    4
6665    4
6666    4
6667    4
6668    4
6669    4
6671    4
Name: product_name, dtype: int64


In [123]:
#check product consistency across multiple product fields

product_consistency_check = df.groupby("product_id").agg({
    "product_name":"nunique",
    "product_category":"nunique",
    "product_brand":"nunique",
    "product_size":"nunique",
    "product_color":"nunique",
    "product_material":"nunique"



}).sort_values(by = "product_name", ascending=False)
print("Product consistency check:")
print(product_consistency_check.head(10))

Product consistency check:
            product_name  product_category  product_brand  product_size  \
product_id                                                                
1                      4                 5              3             3   
6670                   4                 5              3             3   
6663                   4                 5              3             3   
6664                   4                 5              3             3   
6665                   4                 5              3             3   
6666                   4                 5              3             3   
6667                   4                 5              3             3   
6668                   4                 5              3             3   
6669                   4                 5              3             3   
6671                   4                 5              3             3   

            product_color  product_material  
product_id                

### Product ID Consistency Observation

The product_id column is not fully consistent. Several product IDs map to multiple product names, categories, brands, colors, sizes, and materials. Because of this, product_id should not be treated as a clean product table primary key. For this project, product fields will be analyzed as descriptive attributes, and product-level analysis will focus more on product_category, product_brand, and product_name rather than assuming product_id uniquely defines a product.

In [124]:
#Check promotion consistency across multiple promotion details

promotion_consistency_check = df.groupby("promotion_id").agg({
    "promotion_type":"nunique",
    "promotion_channel":"nunique",
    "promotion_target_audience":"nunique",
    "promotion_start_date":"nunique",
    "promotion_end_date":"nunique"

}).sort_values(by = "promotion_type", ascending=False)

print("Promotion consistency check:")
print(promotion_consistency_check.head(10))

Promotion consistency check:
              promotion_type  promotion_channel  promotion_target_audience  \
promotion_id                                                                 
1                          3                  3                          2   
672                        3                  3                          2   
659                        3                  3                          2   
660                        3                  3                          2   
661                        3                  3                          2   
662                        3                  3                          2   
663                        3                  3                          2   
664                        3                  3                          2   
665                        3                  3                          2   
666                        3                  3                          2   

              promotion_start_date

### Promotion ID Consistency Observation

The promotion_id column is not fully consistent. Several promotion IDs map to multiple promotion types, promotion channels, target audiences, start dates, and end dates. Because of this, promotion_id should not be treated as a clean primary key for a promotions table. For this project, promotion fields will be analyzed as descriptive marketing attributes rather than assuming promotion_id uniquely defines a single promotion campaign.

In [125]:
#Check whether store_locatuion maps to store city/state/zip
store_location_consistency_check = df.groupby("store_location").agg({
    "store_city":"nunique",
    "store_state":"nunique",
    "store_zip_code":"nunique"
}).sort_values(by = "store_city", ascending=False)
print("Store location consistency check:")
print(store_location_consistency_check.head(10))

Store location consistency check:
                store_city  store_state  store_zip_code
store_location                                         
Location A               4            3           84396
Location B               4            3           84455
Location C               4            3           84346
Location D               4            3           84403


### Store Location Consistency Observation

The store_location column is not a unique store identifier. Each store location label maps to multiple cities, states, and a very large number of zip codes. Because of this, store_location will be treated as a broad store group or location category rather than a true store-level primary key.

### Key Validation Summary

After checking duplicate values and ID consistency, customer_id appears to be the only reliable unique row identifier in the dataset. The dataset contains 1,000,000 rows and 1,000,000 unique customer IDs, so each row appears to represent one customer-level record.

transaction_id, product_id, promotion_id, and store_location are not reliable primary keys because they either contain duplicate values or map to multiple inconsistent attributes. Therefore, this analysis will be performed primarily at the customer level, while product, promotion, and store fields will be treated as descriptive attributes for segmentation and comparison.

In [126]:
# Check churn distribution and count

df["churned"].value_counts()

churned
No     500271
Yes    499729
Name: count, dtype: int64

In [127]:
#Churn percentage
df["churned"].value_counts(normalize=True)*100

churned
No     50.0271
Yes    49.9729
Name: proportion, dtype: float64

### Churn Distribution Observation

The churned column is nearly balanced, with approximately 50.03% non-churned customers and 49.97% churned customers. This means the predictive modeling task is a balanced binary classification problem. Because the classes are balanced, accuracy can be used as one evaluation metric, but precision, recall, F1-score, confusion matrix, and ROC-AUC should still be reviewed to understand model performance more completely.

### Initial Data Quality and Structure Findings

The dataset contains 1,000,000 rows and 78 columns. The customer_id column is unique for every row, so each row appears to represent one customer-level record.

The transaction_id, product_id, promotion_id, and store_location columns are not reliable primary keys because they contain duplicate values or map to inconsistent attributes. Therefore, this project will use customer_id as the primary row identifier and analyze the dataset primarily at the customer level.

The churn target variable is almost perfectly balanced, making it suitable for classification modeling later in the project.

## Customer Value Analysis

In [128]:
# Create a working copy for analysis
df_clean = df.copy()

#Preview customer value relatd fields
customer_value_col = [
    "customer_id",
    "age",
    "gender",
    "income_bracket",
    "loyalty_program",
    "membership_years",
    "purchase_frequency",
    "avg_purchase_value",
    "avg_transaction_value",
    "total_sales",
    "total_transactions",
    "total_items_purchased",
    "churned"

]


df_clean[customer_value_col].head()


,customer_id,age,gender,income_bracket,loyalty_program,membership_years,purchase_frequency,avg_purchase_value,avg_transaction_value,total_sales,total_transactions,total_items_purchased,churned
0,1,56,Other,High,No,0,Weekly,411.13,171.83,563.16,69,367,No
1,2,69,Female,Medium,No,2,Daily,268.71,20.18,7554.57,8,475,No
2,3,46,Female,Low,No,5,Weekly,246.79,55.17,7564.14,73,138,No
3,4,32,Female,Low,No,0,Weekly,178.92,15.79,8125.92,20,158,No
4,5,60,Female,Low,Yes,7,Yearly,214.06,240.03,114.32,83,263,Yes


In [129]:
#statistically describe customer value related fields
df_clean[customer_value_col].describe()

,customer_id,age,membership_years,avg_purchase_value,avg_transaction_value,total_sales,total_transactions,total_items_purchased
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,500000.500000,48.496605,4.497453,254.886444,255.115768,5056.059765,49.987386,250.042763
std,288675.278933,17.874381,2.872406,141.494923,141.430014,2859.100058,28.571689,143.984546
min,1.000000,18.000000,0.000000,10.000000,10.000000,100.010000,1.000000,1.000000
25%,250000.750000,33.000000,2.000000,132.220000,132.510000,2577.867500,25.000000,125.000000
50%,500000.500000,49.000000,4.000000,254.930000,255.230000,5059.695000,50.000000,250.000000
75%,750000.250000,64.000000,7.000000,377.350000,377.670000,7534.802500,75.000000,375.000000
max,1000000.000000,79.000000,9.000000,500.000000,500.000000,9999.980000,99.000000,499.000000


In [130]:
df_clean[customer_value_col].dtypes

df_clean["purchase_frequency"].head(20)

0      Weekly
1       Daily
2      Weekly
3      Weekly
4      Yearly
5      Weekly
6     Monthly
7     Monthly
8       Daily
9      Yearly
10     Yearly
11      Daily
12     Weekly
13     Yearly
14      Daily
15     Yearly
16     Weekly
17     Yearly
18     Weekly
19     Weekly
Name: purchase_frequency, dtype: object

### Customer Value Statistical Summary

The customer value fields show a complete dataset with no missing values across the selected columns. Total sales range from approximately $100 to $10,000, with a mean and median near $5,056, suggesting a fairly balanced distribution. Customers average about 50 transactions and 250 total items purchased. The average purchase value and average transaction value are very similar, which may indicate these fields measure closely related customer spending behavior. Customer_id is an identifier and should not be interpreted as a numeric business metric.

In [131]:
#create customer value segments based on total sales

df_clean["customer_value_segment"] = pd.qcut(df_clean["total_sales"], q=3, labels=["Low Value", "Medium Value", "High Value"])

df_clean["customer_value_segment"].value_counts()

customer_value_segment
High Value      333334
Low Value       333333
Medium Value    333333
Name: count, dtype: int64

In [132]:
# Summarize customer behavior by value segment
customer_value_summary = df_clean.groupby("customer_value_segment", observed=True).agg({
    "customer_id": "count",
    "total_sales": ["sum", "mean", "median"],
    "avg_transaction_value": "mean",
    "avg_purchase_value": "mean",
    "total_transactions": "mean",
    "total_items_purchased": "mean",
    "membership_years": "mean"
}).round(2)

customer_value_summary

customer_id   total_sales                    \
                             count           sum     mean   median   
customer_value_segment                                               
Low Value                   333333  5.843430e+08  1753.03  1751.56   
Medium Value                333333  1.686306e+09  5058.92  5059.69   
High Value                  333334  2.785411e+09  8356.22  8353.38   

                       avg_transaction_value avg_purchase_value  \
                                        mean               mean   
customer_value_segment                                            
Low Value                             255.13             255.11   
Medium Value                          254.87             254.76   
High Value                            255.34             254.79   

                       total_transactions total_items_purchased  \
                                     mean                  mean   
customer_value_segment                                            
Low Value                           50.00                250.45   
Medium Value                        49.99                249.77   
High Value                          49.98                249.91   

                       membership_years  
                                   mean  
customer_value_segment                   
Low Value                          4.50  
Medium Value                       4.49  
High Value                         4.49

### Customer Value Segment Observation

Customers were split into Low, Medium, and High Value segments based on total_sales. As expected, total sales increases significantly across the three segments. However, average transaction value, average purchase value, total transactions, total items purchased, and membership years remain nearly identical across all value segments.

This suggests that total_sales may not be strongly connected to the other customer behavior fields in this dataset. In a real retail environment, high-value customers would typically be expected to have higher transaction values, more transactions, or more items purchased. Because these patterns are not present, additional validation is needed before using total_sales as the only measure of customer value.

In [133]:
#Check whether total_sales is actually related to otehr value columns

value_corr_cols =[
    "total_sales",
    "avg_transaction_value",
    "avg_purchase_value",
    "total_transactions",
    "total_items_purchased",
    "membership_years"
]

df_clean[value_corr_cols].corr().round(3)

,total_sales,avg_transaction_value,avg_purchase_value,total_transactions,total_items_purchased,membership_years
total_sales,1.000,0.001,-0.001,-0.001,-0.001,-0.002
avg_transaction_value,0.001,1.000,0.001,-0.000,0.000,0.002
avg_purchase_value,-0.001,0.001,1.000,-0.002,0.001,-0.002
total_transactions,-0.001,-0.000,-0.002,1.000,0.000,0.001
total_items_purchased,-0.001,0.000,0.001,0.000,1.000,0.001
membership_years,-0.002,0.002,-0.002,0.001,0.001,1.000


### Correlation Observation

The correlation matrix shows that total_sales has almost no correlation with avg_transaction_value, avg_purchase_value, total_transactions, total_items_purchased, or membership_years. This is unexpected for retail data because total sales would normally be connected to transaction count, average transaction value, and total items purchased.

This suggests that the dataset may be synthetic or that several customer value fields were generated independently. Because of this, total_sales should still be analyzed as a revenue metric, but conclusions about what drives customer value should be made carefully.

In [134]:
#Does total sales match transaction logic

df_clean["expected_sales_from_transactions"] = (
    df_clean["avg_transaction_value"] * df_clean["total_transactions"]
)

df_clean[[
    "total_sales",
    "avg_transaction_value",
    "total_transactions",
    "expected_sales_from_transactions"
]].head(10)

,total_sales,avg_transaction_value,total_transactions,expected_sales_from_transactions
0,563.16,171.83,69,11856.27
1,7554.57,20.18,8,161.44
2,7564.14,55.17,73,4027.41
3,8125.92,15.79,20,315.80
4,114.32,240.03,83,19922.49
5,3372.17,52.14,52,2711.28
6,1322.64,49.28,58,2858.24
7,1716.65,11.62,86,999.32
8,1358.62,383.94,88,33786.72
9,6757.70,134.85,74,9978.90


In [135]:
df_clean["sales_difference"] = (
    df_clean["expected_sales_from_transactions"] - df_clean["total_sales"]
)

df_clean[[
    "total_sales",
    "expected_sales_from_transactions",
    "sales_difference"
]].describe().round(2)

,total_sales,expected_sales_from_transactions,sales_difference
count,1000000.00,1000000.00,1000000.00
mean,5056.06,12750.95,7694.89
std,2859.10,10927.07,11293.40
min,100.01,10.01,-9968.25
25%,2577.87,3722.00,-955.55
50%,5059.70,9665.36,4875.94
75%,7534.80,19339.94,14502.54
max,9999.98,49497.03,49106.85


### Total Sales Validation Observation

To validate the reported total_sales field, I compared it against an expected sales calculation using avg_transaction_value multiplied by total_transactions. The results showed large differences between reported total_sales and calculated expected sales.

The average reported total_sales was approximately $5,056, while the average calculated expected sales was approximately $12,751. This suggests that total_sales is not directly derived from avg_transaction_value and total_transactions. Because of this, total_sales should be treated carefully as a reported revenue field, and a separate calculated customer value metric may be more useful for behavior-based analysis.

In [136]:
#Create a better customer value netric

df_clean["calculated_customer_value"] = (
    df_clean["avg_transaction_value"] * df_clean["total_transactions"])

In [137]:
#Check correlation again

value_corr_cols_updated =[
    "total_sales",
    "calculated_customer_value",
    "avg_transaction_value",
    "avg_purchase_value",
    "total_transactions",
    "total_items_purchased",
    "membership_years"
]

df_clean[value_corr_cols_updated].corr().round(3)


,total_sales,calculated_customer_value,avg_transaction_value,avg_purchase_value,total_transactions,total_items_purchased,membership_years
total_sales,1.000,0.001,0.001,-0.001,-0.001,-0.001,-0.002
calculated_customer_value,0.001,1.000,0.647,-0.002,0.667,0.000,0.002
avg_transaction_value,0.001,0.647,1.000,0.001,-0.000,0.000,0.002
avg_purchase_value,-0.001,-0.002,0.001,1.000,-0.002,0.001,-0.002
total_transactions,-0.001,0.667,-0.000,-0.002,1.000,0.000,0.001
total_items_purchased,-0.001,0.000,0.000,0.001,0.000,1.000,0.001
membership_years,-0.002,0.002,0.002,-0.002,0.001,0.001,1.000


In [138]:
#Create cistomer value segments based on calculated customer value

df_clean["calculated_value_segment"] = pd.qcut(df_clean["calculated_customer_value"], q=3, labels=["Low Calculated Value", "Medium Calculated Value", "High Calculated Value"])

df_clean["calculated_value_segment"].value_counts()

calculated_value_segment
High Calculated Value      333334
Low Calculated Value       333333
Medium Calculated Value    333333
Name: count, dtype: int64

In [139]:
caluclated_value_summary = df_clean.groupby("calculated_value_segment", observed=True).agg({

    "customer_id": "count",
    "total_sales": ["sum", "mean", "median"],
    "calculated_customer_value": ["sum", "mean", "median"],
    "avg_transaction_value": "mean",
    "avg_purchase_value": "mean",
    "total_transactions": "mean",
    "total_items_purchased": "mean",
    "membership_years": "mean"
}).round(2)

caluclated_value_summary

customer_id   total_sales                    \
                               count           sum     mean   median   
calculated_value_segment                                               
Low Calculated Value          333333  1.684118e+09  5052.36  5047.28   
Medium Calculated Value       333333  1.685087e+09  5055.27  5061.02   
High Calculated Value         333334  1.686854e+09  5060.55  5072.60   

                         calculated_customer_value                      \
                                               sum      mean    median   
calculated_value_segment                                                 
Low Calculated Value                  8.148176e+08   2444.46   2292.93   
Medium Calculated Value               3.310101e+09   9930.31   9665.35   
High Calculated Value                 8.626031e+09  25878.04  24246.10   

                         avg_transaction_value avg_purchase_value  \
                                          mean               mean   
calculated_value_segment                                            
Low Calculated Value                    153.93             255.08   
Medium Calculated Value                 246.84             255.02   
High Calculated Value                   364.58             254.56   

                         total_transactions total_items_purchased  \
                                       mean                  mean   
calculated_value_segment                                            
Low Calculated Value                  28.41                249.94   
Medium Calculated Value               49.06                250.12   
High Calculated Value                 72.48                250.07   

                         membership_years  
                                     mean  
calculated_value_segment                   
Low Calculated Value                 4.50  
Medium Calculated Value              4.49  
High Calculated Value                4.50

### Calculated Customer Value Observation

Because the reported total_sales column did not align with avg_transaction_value and total_transactions, I created a calculated_customer_value field using avg_transaction_value multiplied by total_transactions.

This calculated metric produced more logical customer value segments. High calculated value customers had both higher average transaction values and more total transactions, while low calculated value customers had lower transaction values and fewer transactions.

However, reported total_sales remained nearly the same across the calculated value segments. This confirms that total_sales is not strongly connected to the behavioral purchase fields in this dataset. For behavior-based customer value analysis, calculated_customer_value is a stronger metric than the original total_sales field.

## Customer Value and Churn Analysis

After creating a behavior-based customer value metric, I analyzed whether customer value is associated with churn. This helps determine whether high-value customers are more or less likely to leave and whether retention strategies should prioritize specific value segments.

In [140]:
churn_by_calculated_value = df_clean.groupby("calculated_value_segment", observed=True).agg({
    "customer_id": "count",
    "churned": lambda x: (x == "Yes").mean() * 100,
    "calculated_customer_value": "mean",
    "avg_transaction_value": "mean",
    "total_transactions": "mean"
}).round(2)

churn_by_calculated_value

,customer_id,churned,calculated_customer_value,avg_transaction_value,total_transactions
calculated_value_segment,,,,,
Low Calculated Value,333333,49.98,2444.46,153.93,28.41
Medium Calculated Value,333333,50.00,9930.31,246.84,49.06
High Calculated Value,333334,49.94,25878.04,364.58,72.48


### Churn by Calculated Value Segment Observation

Churn rates were nearly identical across Low, Medium, and High Calculated Value segments, staying close to 50% in each group. This suggests that behavior-based customer value, as measured by avg_transaction_value multiplied by total_transactions, does not meaningfully explain churn in this dataset.

Although high calculated value customers have much higher transaction values and more transactions, they are not significantly less likely to churn. This means retention analysis should look beyond customer value and examine engagement, recency, support calls, email subscriptions, app usage, website visits, and other behavioral indicators.

In [141]:
loyalty_value_churn_summary = df_clean.groupby("loyalty_program").agg({
    "customer_id": "count",
    "churned": lambda x: (x == "Yes").mean() * 100,
    "calculated_customer_value": ["mean","median"],
    "avg_transaction_value": "mean",
    "total_transactions": "mean"
}).round(2)

loyalty_value_churn_summary

customer_id  churned calculated_customer_value           \
                      count <lambda>                      mean   median   
loyalty_program                                                           
No                   500288    49.99                  12741.45  9624.98   
Yes                  499712    49.96                  12760.46  9703.67   

                avg_transaction_value total_transactions  
                                 mean               mean  
loyalty_program                                           
No                             254.86              49.97  
Yes                            255.37              50.00

### Loyalty Program Observation

Loyalty program members and non-members showed nearly identical churn rates and customer value metrics. Both groups had churn rates close to 50%, average calculated customer values around $12,750, and average transaction counts near 50.

This suggests that loyalty program membership, by itself, does not appear to be a strong differentiator of customer value or churn risk in this dataset. Additional analysis is needed to determine whether other behavioral factors are more useful for identifying churn risk.

In [142]:
loyalty_by_value_segment = pd.crosstab(
    df_clean["loyalty_program"],
    df_clean["calculated_value_segment"],
    normalize="index"
) * 100

loyalty_by_value_segment.round(2)

calculated_value_segment,Low Calculated Value,Medium Calculated Value,High Calculated Value
loyalty_program,,,
No,33.42,33.26,33.32
Yes,33.25,33.41,33.34


### Loyalty Program by Value Segment Observation

Loyalty program members and non-members were almost evenly distributed across Low, Medium, and High Calculated Value segments. This means loyalty program participation is not strongly associated with belonging to a higher customer value segment in this dataset.

In [143]:
income_value_churn_summary = df_clean.groupby("income_bracket").agg({
    "customer_id": "count",
    "churned": lambda x: (x == "Yes").mean() * 100,
    "calculated_customer_value": "mean",
    "avg_transaction_value": "mean",
    "total_transactions": "mean"
}).round(2)

income_value_churn_summary

,customer_id,churned,calculated_customer_value,avg_transaction_value,total_transactions
income_bracket,,,,,
High,333612,49.94,12727.61,254.65,49.97
Low,333021,49.94,12770.91,255.32,50.03
Medium,333367,50.04,12754.37,255.38,49.97


In [144]:
income_by_value_segment = pd.crosstab(
    df_clean["income_bracket"],
    df_clean["calculated_value_segment"],
    normalize="index"
) * 100

income_by_value_segment.round(2)

calculated_value_segment,Low Calculated Value,Medium Calculated Value,High Calculated Value
income_bracket,,,
High,33.47,33.24,33.28
Low,33.22,33.39,33.40
Medium,33.31,33.37,33.32


### Income Bracket Observation

Income brackets showed very similar churn rates and customer value patterns across groups. This suggests that income level does not appear to meaningfully explain churn or customer value differences in this dataset. Since customer value, loyalty membership, and income bracket all show weak relationships with churn, the next step is to examine behavioral and engagement-related variables.

In [145]:
# Churn rate by purchase frequency\

purchase_frequency_churn = df_clean.groupby("purchase_frequency").agg({
    "customer_id": "count",
    "churned": lambda x: (x == "Yes").mean() * 100,
    "calculated_customer_value": "mean",
    "avg_transaction_value": "mean",
    "total_transactions": "mean"
}).round(2)

purchase_frequency_churn

,customer_id,churned,calculated_customer_value,avg_transaction_value,total_transactions
purchase_frequency,,,,,
Daily,249533,49.95,12739.09,255.09,49.98
Monthly,249932,49.80,12759.03,254.91,50.03
Weekly,249768,50.08,12784.67,255.31,50.08
Yearly,250767,50.06,12721.11,255.16,49.86


In [146]:
# Purchase frequency distribution by calculated value segment
purchase_frequency_by_value = pd.crosstab(
    df_clean["purchase_frequency"],
    df_clean["calculated_value_segment"],
    normalize="index"   

)*100

purchase_frequency_by_value.round(2)

calculated_value_segment,Low Calculated Value,Medium Calculated Value,High Calculated Value
purchase_frequency,,,
Daily,33.31,33.38,33.31
Monthly,33.27,33.34,33.39
Weekly,33.36,33.18,33.46
Yearly,33.40,33.43,33.17


In [147]:
# Create a churn frequency score
frequency_map ={
    "Yearly": 1,
    "Monthly": 2,
    "Weekly": 3,
    "Daily" : 4
}

df_clean["purchase_frequency_score"] = df_clean["purchase_frequency"].map(frequency_map)
df_clean[["purchase_frequency", "purchase_frequency_score"]].head(10)

,purchase_frequency,purchase_frequency_score
0,Weekly,3
1,Daily,4
2,Weekly,3
3,Weekly,3
4,Yearly,1
5,Weekly,3
6,Monthly,2
7,Monthly,2
8,Daily,4
9,Yearly,1


In [148]:
# Check whether scored differ by churn status

frequency_score_by_churn = df_clean.groupby("churned").agg({
    "purchase_frequency_score": "mean",
    "calculated_customer_value": "mean",
    "avg_transaction_value": "mean",
    "total_transactions": "mean"
}).round(2)
frequency_score_by_churn

,purchase_frequency_score,calculated_customer_value,avg_transaction_value,total_transactions
churned,,,,
No,2.5,12751.02,255.11,50.01
Yes,2.5,12750.88,255.12,49.96


### Purchase Frequency Observation

Purchase frequency did not show a strong relationship with churn. Churn rates remained close to 50% across Daily, Weekly, Monthly, and Yearly customers. Purchase frequency was also evenly distributed across Low, Medium, and High Calculated Value segments, with each value group showing roughly one-third distribution across categories.

This suggests that purchase frequency alone does not meaningfully explain churn in this dataset. The next step is to examine behavioral and engagement variables such as days_since_last_purchase, app_usage, website_visits, social_media_engagement, email_subscriptions, and customer_support_calls.

## Engagement and Recency Analysis

Since customer value, loyalty membership, income bracket, and purchase frequency did not show strong differences in churn, the next step is to examine behavioral engagement and recency fields. These variables may provide better insight into whether customer activity level is associated with churn.

In [149]:
# Create engagement columns list
engagement_cols = [
    "churned",
    "days_since_last_purchase",
    "customer_support_calls",
    "email_subscriptions",
    "app_usage",
    "website_visits",
    "social_media_engagement"
]

df_clean[engagement_cols].head()

,churned,days_since_last_purchase,customer_support_calls,email_subscriptions,app_usage,website_visits,social_media_engagement
0,No,40,5,No,High,30,High
1,No,338,6,No,High,40,Medium
2,No,61,2,Yes,Low,89,Medium
3,No,42,12,No,Low,12,Low
4,Yes,242,3,No,Medium,31,Low


In [150]:
#Check data types before aggregation, numerical or categorical
df_clean[engagement_cols].dtypes

churned                     object
days_since_last_purchase     int64
customer_support_calls       int64
email_subscriptions         object
app_usage                   object
website_visits               int64
social_media_engagement     object
dtype: object

In [151]:
# Statistical summary of engagement metrics by churn status
engagement_summary_by_churn = df_clean.groupby("churned").agg({
    "days_since_last_purchase": "mean",
    "customer_support_calls": "mean",
    "website_visits": "mean",
}).round(2)

engagement_summary_by_churn

,days_since_last_purchase,customer_support_calls,website_visits
churned,,,
No,182.21,9.49,49.54
Yes,181.85,9.50,49.48


### Numeric Engagement and Recency Observation

After comparing numeric engagement and recency fields by churn status, the averages were nearly identical for churned and non-churned customers. Non-churned customers averaged 182.21 days since last purchase, 9.49 support calls, and 49.54 website visits, while churned customers averaged 181.85 days since last purchase, 9.50 support calls, and 49.48 website visits.

These small differences suggest that days_since_last_purchase, customer_support_calls, and website_visits do not meaningfully explain churn in this dataset. Combined with earlier findings, this supports the possibility that the churn label may be synthetic or weakly connected to the available customer behavior variables.

In [152]:
# We removed categorical col for numeric analysis, but we can also check distribution of categorical engagement metrics by churn status

engagement_categorical_cols = [
    "email_subscriptions",
    "app_usage",
    "social_media_engagement"
]

for col in engagement_categorical_cols:
    print(f"\nCHurn rate by {col}:\n")
    churn_rate = df_clean.groupby(col)["churned"].apply(
        lambda x:(x == "Yes").mean() * 100
    ).round(2)
    print(churn_rate)



CHurn rate by email_subscriptions:

email_subscriptions
No     49.90
Yes    50.04
Name: churned, dtype: float64

CHurn rate by app_usage:

app_usage
High      49.92
Low       49.97
Medium    50.02
Name: churned, dtype: float64

CHurn rate by social_media_engagement:

social_media_engagement
High      49.95
Low       49.98
Medium    49.99
Name: churned, dtype: float64


In [153]:
#Check full percentage split of engagement categorical metrics

for col in engagement_categorical_cols:
    print(F"\nChurn distribution by {col}:\n")
    display(
        pd.crosstab(
            df_clean[col],
            df_clean["churned"],
            normalize="index"
        ).round(4) * 100 
    )


Churn distribution by email_subscriptions:



churned,No,Yes
email_subscriptions,,
No,50.10,49.90
Yes,49.96,50.04



Churn distribution by app_usage:



churned,No,Yes
app_usage,,
High,50.08,49.92
Low,50.03,49.97
Medium,49.98,50.02



Churn distribution by social_media_engagement:



churned,No,Yes
social_media_engagement,,
High,50.05,49.95
Low,50.02,49.98
Medium,50.01,49.99


In [154]:
# Check my intial conclusion that churn is not correlated with engagement metrics by looking at distribution of engagement metrics across value segments

df_clean["churned_flag"] = df_clean["churned"].map({"Yes":1, "No":0})

engagement_corr_cols =[
    "churned_flag",
    "days_since_last_purchase",
    "customer_support_calls",
    "website_visits"


]

df_clean[engagement_corr_cols].corr().round(3)

,churned_flag,days_since_last_purchase,customer_support_calls,website_visits
churned_flag,1.000,-0.002,0.001,-0.001
days_since_last_purchase,-0.002,1.000,0.001,-0.001
customer_support_calls,0.001,0.001,1.000,0.002
website_visits,-0.001,-0.001,0.002,1.000


### Engagement and Churn Observation

Numeric engagement and recency variables showed almost no relationship with churn. The correlation between churned_flag and days_since_last_purchase, customer_support_calls, and website_visits was close to zero.

Categorical engagement variables also showed nearly identical churn distributions across groups. Email subscription status, app usage level, and social media engagement level all had churn rates close to 50%.

These findings suggest that the churn label is weakly connected to the available customer behavior variables. Combined with earlier analysis, this supports the possibility that churn was synthetically or randomly generated in this dataset. Because of this, any predictive churn model may have limited performance unless additional meaningful features are available.

In [155]:
# To confirm that churn doesnt have any relationship with any column in the dataset, we can check correlation of churned flag with all  columns
df_clean["churned_flag"] = df_clean["churned"].map({"Yes":1, "No":0})



In [156]:
# Now we scan the numeric columns for churn correlations

numeric_cols = df_clean.select_dtypes(include=["int64", "Float64"]).columns.tolist()

exclude_numeric_cols = [
    "customer_id",
    "transaction_id",
    "product_id",
    "promotion_id",
    "store_zip_code",
    "churned_flag"


]

numeric_cols_for_scan = [
    col for col in numeric_cols if col not in exclude_numeric_cols

]

#Correlation with churn
numeric_churn_corr =(
    df_clean[numeric_cols_for_scan + ["churned_flag"]].corr()["churned_flag"].drop("churned_flag").round(3).sort_values(ascending=False)


)

print("Correlation of numeric features with churned flag:\n")
print(numeric_churn_corr)


Correlation of numeric features with churned flag:

product_return_rate                 0.003
total_items_purchased               0.002
week_of_year                        0.001
max_single_purchase_value           0.001
min_single_purchase_value           0.001
product_shelf_life                  0.001
avg_spent_per_category              0.001
customer_support_calls              0.001
number_of_children                  0.001
total_discounts_received            0.000
expected_sales_from_transactions   -0.000
sales_difference                    0.000
product_stock                      -0.000
product_review_count                0.000
product_rating                     -0.000
calculated_customer_value          -0.000
customer_zip_code                   0.000
age                                -0.000
purchase_frequency_score           -0.000
avg_purchase_value                 -0.000
quantity                            0.000
total_returned_value               -0.000
total_returned_items    

In [157]:
categorical_cols = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()

exclude_categorical_cols = [
    "churned"
]

categorical_cols_for_scan = [
    col for col in categorical_cols
    if col not in exclude_categorical_cols
]

cat_churn_results = []

for col in categorical_cols_for_scan:
    unique_count = df_clean[col].nunique()
    
    # Avoid high-cardinality columns like product names if too many unique values
    if unique_count <= 30:
        churn_rates = df_clean.groupby(col)["churned_flag"].mean() * 100
        churn_spread = churn_rates.max() - churn_rates.min()
        
        cat_churn_results.append({
            "column": col,
            "unique_values": unique_count,
            "min_churn_rate": churn_rates.min(),
            "max_churn_rate": churn_rates.max(),
            "churn_rate_spread": churn_spread
        })

cat_churn_signal = pd.DataFrame(cat_churn_results).sort_values(
    by="churn_rate_spread",
    ascending=False
)

cat_churn_signal.round(2).head(20)

C:\Users\giova\AppData\Local\Temp\ipykernel_74556\1353992973.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  churn_rates = df_clean.groupby(col)["churned_flag"].mean() * 100


,column,unique_values,min_churn_rate,max_churn_rate,churn_rate_spread
15,product_color,5,49.77,50.19,0.42
21,customer_city,4,49.84,50.14,0.30
16,product_material,4,49.82,50.11,0.29
24,store_state,3,49.83,50.12,0.29
10,purchase_frequency,4,49.80,50.08,0.29
6,product_category,5,49.84,50.11,0.26
9,day_of_week,7,49.83,50.09,0.26
18,promotion_effectiveness,3,49.89,50.14,0.25
22,customer_state,3,49.83,50.07,0.24
8,store_location,4,49.83,50.05,0.23


### Churn Signal Scan Conclusion

After analyzing churn across customer value segments, loyalty program status, income brackets, purchase frequency, engagement variables, product attributes, promotion attributes, and location fields, churn rates remained close to 50% across nearly every group.

The broad categorical churn scan showed very small churn-rate spreads. The largest observed spread was only about 0.42 percentage points, which is not large enough to suggest a meaningful business relationship. Numeric correlation analysis also showed near-zero correlation between churn and available behavioral variables.

Based on these results, the churned column appears to be synthetic or weakly connected to the available dataset features. Because of this, churn should not be the main focus of the business analysis. Predictive modeling can still be performed later as a cautionary exercise, but model performance may be close to baseline because the target variable does not show strong explanatory signal.

## Project Direction Update After Data Validation

After validating the dataset, the project direction shifted. The original goal was to perform customer value, churn, product, and promotion analysis. However, several validation checks showed that the dataset is highly synthetic and evenly distributed.

Key validation findings included:

- customer_id was the only reliable unique row identifier.
- transaction_id was not reliable as a transaction key because repeated transaction IDs were linked to multiple customers, products, dates, and sales values.
- product_id, promotion_id, and store_location were not reliable clean primary keys.
- total_sales did not align with avg_transaction_value and total_transactions.
- churn stayed close to 50% across nearly all customer, product, promotion, income, loyalty, frequency, and engagement groups.

Because of these findings, the project was reframed as a data validation, feature engineering, SQL table modeling, and reporting practice project rather than a project focused on strong business recommendations or churn prediction.

## Product and Promotion Attribute Analysis

### Product and Promotion Analysis Note

Because transaction_id, product_id, and promotion_id were not reliable unique keys, this section should not be interpreted as true transaction-level product performance or campaign effectiveness analysis. Instead, product and promotion fields are used as descriptive customer-level attributes.

The goal of this section is to practice grouped analysis and SQL-style reporting by comparing customer value, transaction behavior, return rates, ratings, discounts, and churn across product categories, brands, and promotion types. Any business conclusions should be stated carefully because the dataset appears synthetic and evenly distributed.

In [158]:
#  Product category performance

product_category_summary = df_clean.groupby("product_category").agg({
    "customer_id": "count",
    "calculated_customer_value": ["sum", "mean", "median"],
    "avg_transaction_value": "mean",
    "total_transactions": "mean",
    "total_items_purchased": "mean",
    "product_return_rate": "mean",
    "product_rating": "mean",
    "churned": lambda x: (x == "Yes").mean() * 100
})

product_category_summary.round(2).sort_values(by = ("calculated_customer_value", "sum"), ascending=False).head(20)

customer_id calculated_customer_value                     \
                       count                       sum      mean   median   
product_category                                                            
Toys                  200669              2.559170e+09  12753.19  9684.48   
Groceries             200214              2.555058e+09  12761.63  9667.70   
Clothing              199778              2.552960e+09  12778.98  9686.67   
Furniture             199583              2.544757e+09  12750.37  9635.44   
Electronics           199756              2.539005e+09  12710.53  9649.68   

                 avg_transaction_value total_transactions  \
                                  mean               mean   
product_category                                            
Toys                            254.90              50.03   
Groceries                       255.40              49.95   
Clothing                        255.48              50.01   
Furniture                       254.98              49.98   
Electronics                     254.82              49.96   

                 total_items_purchased product_return_rate product_rating  \
                                  mean                mean           mean   
product_category                                                            
Toys                            249.77                0.25            3.0   
Groceries                       250.13                0.25            3.0   
Clothing                        249.87                0.25            3.0   
Furniture                       250.51                0.25            3.0   
Electronics                     249.94                0.25            3.0   

                  churned  
                 <lambda>  
product_category           
Toys                49.84  
Groceries           50.01  
Clothing            49.93  
Furniture           50.11  
Electronics         49.97

### Product Category Performance Observation

Toys and Groceries generated the highest total calculated customer value, mainly because they had slightly higher customer counts than the other categories. Clothing had the highest average calculated customer value, but the difference between categories was very small.

Product return rates, product ratings, and churn rates were nearly identical across all product categories. This suggests that product_category provides limited differentiation for return behavior, customer satisfaction, or churn in this dataset. Product category can still be used for dashboard segmentation, but the business insights should be stated carefully because the category-level differences are small.

In [159]:
# Now we can check whether product brand shows stronger differences than category

product_brand_summary = df_clean.groupby("product_brand").agg({
    "customer_id": "count",
    "calculated_customer_value": ["sum", "mean", "median"],
    "avg_transaction_value": "mean",
    "total_transactions": "mean",
    "total_items_purchased": "mean",
    "product_return_rate": "mean",
    "product_rating": "mean",
    "churned": lambda x: (x == "Yes").mean() * 100
})

product_brand_summary.round(2).sort_values(by = ("calculated_customer_value", "sum"), ascending=False).head(20)



customer_id calculated_customer_value                     \
                    count                       sum      mean   median   
product_brand                                                            
Brand Y            333775              4.262104e+09  12769.39  9684.18   
Brand Z            333608              4.245890e+09  12727.18  9634.24   
Brand X            332617              4.242956e+09  12756.28  9672.00   

              avg_transaction_value total_transactions total_items_purchased  \
                               mean               mean                  mean   
product_brand                                                                  
Brand Y                      255.22              50.05                249.68   
Brand Z                      254.92              49.93                250.14   
Brand X                      255.22              49.98                250.31   

              product_return_rate product_rating  churned  
                             mean           mean <lambda>  
product_brand                                              
Brand Y                      0.25            3.0    49.97  
Brand Z                      0.25            3.0    50.02  
Brand X                      0.25            3.0    49.93

### Product Brand Performance Observation

Brand Y generated the highest total calculated customer value and had the highest customer count. Brand Y also had the highest average calculated customer value, followed by Brand X and Brand Z. However, the differences were small across all three brands.

Product return rates, product ratings, and churn rates were nearly identical across brands. This suggests that product_brand does not strongly differentiate customer value, satisfaction, return behavior, or churn in this dataset.

In [160]:
#Now we can check promotion performance

promotion_summary = df_clean.groupby("promotion_type").agg({
    "customer_id": "count", 
    "calculated_customer_value": ["sum", "mean", "median"],
    "avg_transaction_value": "mean",        
    "total_transactions": "mean",
    "avg_discount_used": "mean",
    "total_discounts_received": "mean",
    "churned": lambda x: (x == "Yes").mean() *
    100
}).round(2)

promotion_summary.sort_values(by = ("calculated_customer_value", "sum"), ascending=False).head(20)

customer_id calculated_customer_value                     \
                           count                       sum      mean   median   
promotion_type                                                                  
Buy One Get One Free      333520              4.260734e+09  12775.05  9698.86   
20% Off                   333712              4.251185e+09  12739.08  9651.25   
Flash Sale                332768              4.239032e+09  12738.70  9645.29   

                     avg_transaction_value total_transactions  \
                                      mean               mean   
promotion_type                                                  
Buy One Get One Free                255.33              50.06   
20% Off                             254.87              50.00   
Flash Sale                          255.14              49.90   

                     avg_discount_used total_discounts_received  churned  
                                  mean                     mean <lambda>  
promotion_type                                                            
Buy One Get One Free              0.25                   500.24    49.96  
20% Off                           0.25                   499.71    49.99  
Flash Sale                        0.25                   499.08    49.97

### Promotion Performance Observation

Buy One Get One Free had the highest total and average calculated customer value among promotion types, followed by 20% Off and Flash Sale. However, the differences between promotion types were very small. Average transaction value, total transactions, average discount used, total discounts received, and churn rates were nearly identical across promotion groups.

This suggests that promotion_type does not strongly differentiate customer value or churn in this dataset. Promotion performance can still be included in the dashboard, but insights should be stated carefully because the dataset appears highly balanced across promotion categories.

## Overall Dataset Assessment

After completing dataset structure validation, customer value analysis, churn analysis, engagement analysis, product category analysis, product brand analysis, and promotion analysis, the dataset appears to be highly synthetic and evenly distributed.

Several key findings support this:

- customer_id is the only reliable unique row identifier.
- transaction_id, product_id, promotion_id, and store_location are not reliable clean primary keys.
- total_sales does not correlate with avg_transaction_value, total_transactions, or total_items_purchased.
- calculated_customer_value provides a more logical behavior-based value metric.
- churn remains close to 50% across customer value segments, loyalty program status, income brackets, purchase frequency, engagement variables, product categories, product brands, and promotion types.
- Product ratings, return rates, promotion metrics, and category performance are very evenly distributed.

Because of this, the dataset should not be used to make strong business claims about churn, customer loyalty, product performance, or promotion effectiveness. However, it is still useful for practicing data cleaning, feature engineering, SQL table creation, dashboard preparation, and predictive modeling workflows.

The most valuable insight from this analysis is the importance of validating dataset quality before building dashboards or machine learning models.

## Final Dataset Cleaning and SQL Preparation

After completing the exploratory analysis and validation process, the dataset was found to be highly synthetic and evenly distributed across many fields. Several key variables did not behave like they would in a clean real-world retail dataset.

The main validation findings were:

* `customer_id` was the only reliable unique row identifier.
* `transaction_id` was not reliable as a transaction-level primary key because repeated transaction IDs were linked to multiple customers, products, dates, and sales values.
* `product_id`, `promotion_id`, and `store_location` were not reliable clean keys for normalized product, promotion, or store tables.
* `total_sales` did not align with `avg_transaction_value` and `total_transactions`.
* The `churned` field remained close to 50% across nearly all customer, product, promotion, income, loyalty, frequency, and engagement groups.
* Numeric correlation and categorical churn scans showed little to no meaningful relationship between churn and the available features.

Because of these findings, the dataset should not be used to make strong business claims about churn, customer loyalty, product performance, promotion effectiveness, or sales drivers. Instead, the dataset will be used from this point forward as a practice dataset for data cleaning, SQL-style table creation, SQL querying, reporting preparation, and Excel/Sheets dashboard practice.


## Purpose of the Final Cleaned Dataset

The purpose of this final cleaning step is to prepare a stable version of the dataset for SQL practice and reporting exports.

Although the dataset has limitations, it is still useful for practicing an analyst workflow:

1. Creating a cleaned working dataset.
2. Engineering useful fields such as `churned_flag`, `calculated_customer_value`, and `calculated_value_segment`.
3. Converting date columns into proper datetime format.
4. Creating date-based reporting fields such as year, month, year-month, and quarter.
5. Splitting the flat dataset into SQL-style analytical tables.
6. Practicing joins, grouping, aggregations, CASE statements, CTEs, and window functions.
7. Exporting summary tables for Excel or Google Sheets dashboard practice.

The final cleaned dataset will be treated as a structured practice dataset rather than a source for major business recommendations.


In [161]:
# Create the final working copy for sql analysis
df_final = df_clean.copy()

#Confirm the shaope
df_final.shape

df_final.head()

,customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,...,website_visits,social_media_engagement,days_since_last_purchase,customer_value_segment,expected_sales_from_transactions,sales_difference,calculated_customer_value,calculated_value_segment,purchase_frequency_score,churned_flag
0,1,56,Other,High,No,0,No,Divorced,3,Bachelor's,...,30,High,40,Low Value,11856.27,11293.11,11856.27,Medium Calculated Value,3,0
1,2,69,Female,Medium,No,2,No,Married,2,PhD,...,40,Medium,338,High Value,161.44,-7393.13,161.44,Low Calculated Value,4,0
2,3,46,Female,Low,No,5,No,Married,3,Bachelor's,...,89,Medium,61,High Value,4027.41,-3536.73,4027.41,Low Calculated Value,3,0
3,4,32,Female,Low,No,0,No,Divorced,2,Master's,...,12,Low,42,High Value,315.80,-7810.12,315.80,Low Calculated Value,3,0
4,5,60,Female,Low,Yes,7,Yes,Divorced,2,Bachelor's,...,31,Low,242,Low Value,19922.49,19808.17,19922.49,High Calculated Value,1,1


In [162]:
#Convert date columns
date_cols = [
    "transaction_date",
    "promotion_start_date",
    "promotion_end_date",
    "last_purchase_date",
    "product_manufacture_date",
    "product_expiry_date",


]

for col in date_cols:
    df_final[col] = pd.to_datetime(df_final[col], errors="coerce")


df_final[date_cols].dtypes

transaction_date            datetime64[ns]
promotion_start_date        datetime64[ns]
promotion_end_date          datetime64[ns]
last_purchase_date          datetime64[ns]
product_manufacture_date    datetime64[ns]
product_expiry_date         datetime64[ns]
dtype: object

In [163]:
# Now we can create useful date features for sql analysis
df_final["transaction_year"] = df_final["transaction_date"].dt.year
df_final["transaction_month"] = df_final["transaction_date"].dt.month
df_final["transaction_year_month"] = df_final["transaction_date"].dt.to_period("M").astype(str)
df_final["transaction_quarter"] = df_final["transaction_date"].dt.to_period("Q").astype(str)


df_final[[
    "transaction_date",
    "transaction_year",
    "transaction_month",
    "transaction_year_month",
    "transaction_quarter"
]].head()


,transaction_date,transaction_year,transaction_month,transaction_year_month,transaction_quarter
0,2020-10-11 10:08:52,2020,10,2020-10,2020Q4
1,2021-12-08 01:07:40,2021,12,2021-12,2021Q4
2,2020-02-17 09:40:48,2020,2,2020-02,2020Q1
3,2020-08-13 00:43:14,2020,8,2020-08,2020Q3
4,2021-07-02 11:59:03,2021,7,2021-07,2021Q3


## SQL-Style Analytical Table Creation

Since the original dataset is a single flat file, this section creates SQL-style analytical tables to support interview-focused SQL practice.

These tables are not meant to represent a perfectly normalized production database because several IDs in the dataset were found to be unreliable. Instead, the tables are designed for analytical practice and reporting workflows.

The tables created in this section are:

* `customers`: customer demographics, churn status, and profile fields.
* `customer_behavior`: purchase frequency, recency, support calls, engagement, and shopping behavior fields.
* `purchase_activity`: purchase, transaction, date, payment, quantity, price, and calculated customer value fields.
* `product_attributes`: product category, brand, rating, stock, return, size, color, and material fields.
* `promotion_activity`: promotion type, channel, audience, dates, effectiveness, and discount fields.
* `location_attributes`: store location, store geography, distance, season, holiday, and weekend fields.

These tables will be used to practice SQL joins and business reporting queries. Since `customer_id` is the only reliable unique identifier, most joins will use `customer_id` as the linking key.


In [164]:
# Create customers table for sql
customers = df_final[[
    "customer_id",
    "age",
    "gender",
    "income_bracket",
    "loyalty_program",
    "membership_years",
    "marital_status",
    "education_level",
    "occupation",
    "customer_city",
    "customer_state",
    "customer_zip_code",
    "churned",
    "churned_flag",
    "number_of_children",

]].copy()

customers.head()
    

,customer_id,age,gender,income_bracket,loyalty_program,membership_years,marital_status,education_level,occupation,customer_city,customer_state,customer_zip_code,churned,churned_flag,number_of_children
0,1,56,Other,High,No,0,Divorced,Bachelor's,Self-Employed,City D,State Y,37848,No,0,3
1,2,69,Female,Medium,No,2,Married,PhD,Unemployed,City A,State X,44896,No,0,2
2,3,46,Female,Low,No,5,Married,Bachelor's,Self-Employed,City B,State X,11816,No,0,3
3,4,32,Female,Low,No,0,Divorced,Master's,Employed,City A,State Y,78604,No,0,2
4,5,60,Female,Low,Yes,7,Divorced,Bachelor's,Employed,City B,State Z,17760,Yes,1,2


In [165]:
#Create customer behavior table for sql

customer_behavior = df_final[[
    "customer_id",
    "purchase_frequency",
    "days_since_last_purchase",
    "customer_support_calls",
    "email_subscriptions",
    "app_usage",
    "website_visits",
    "social_media_engagement",
    "online_purchases",
    "in_store_purchases",
    "preferred_store"
]].copy()

customer_behavior.head()

,customer_id,purchase_frequency,days_since_last_purchase,customer_support_calls,email_subscriptions,app_usage,website_visits,social_media_engagement,online_purchases,in_store_purchases,preferred_store
0,1,Weekly,40,5,No,High,30,High,55,86,Location A
1,2,Daily,338,6,No,High,40,Medium,48,2,Location C
2,3,Weekly,61,2,Yes,Low,89,Medium,16,45,Location B
3,4,Weekly,42,12,No,Low,12,Low,50,47,Location B
4,5,Yearly,242,3,No,Medium,31,Low,48,42,Location B


In [166]:
# create purchase activity table for sql analysis
purchase_activity = df_final[[
    "customer_id",
    "transaction_id",
    "transaction_date",
    "transaction_year",
    "transaction_month",
    "transaction_year_month",
    "transaction_quarter",
    "transaction_hour",
    "day_of_week",
    "week_of_year",
    "month_of_year",
    "payment_method",
    "quantity",
    "unit_price",
    "discount_applied",
    "avg_purchase_value",
    "avg_transaction_value",
    "total_sales",
    "total_transactions",
    "total_items_purchased",
    "calculated_customer_value",
    "calculated_value_segment"
]].copy()

purchase_activity.head()

,customer_id,transaction_id,transaction_date,transaction_year,transaction_month,transaction_year_month,transaction_quarter,transaction_hour,day_of_week,week_of_year,...,quantity,unit_price,discount_applied,avg_purchase_value,avg_transaction_value,total_sales,total_transactions,total_items_purchased,calculated_customer_value,calculated_value_segment
0,1,503290,2020-10-11 10:08:52,2020,10,2020-10,2020Q4,18,Wednesday,27,...,8,49.72,0.50,411.13,171.83,563.16,69,367,11856.27,Medium Calculated Value
1,2,347796,2021-12-08 01:07:40,2021,12,2021-12,2021Q4,15,Friday,20,...,7,817.76,0.32,268.71,20.18,7554.57,8,475,161.44,Low Calculated Value
2,3,493688,2020-02-17 09:40:48,2020,2,2020-02,2020Q1,9,Saturday,35,...,8,270.30,0.35,246.79,55.17,7564.14,73,138,4027.41,Low Calculated Value
3,4,861348,2020-08-13 00:43:14,2020,8,2020-08,2020Q3,13,Friday,42,...,2,547.84,0.10,178.92,15.79,8125.92,20,158,315.80,Low Calculated Value
4,5,535835,2021-07-02 11:59:03,2021,7,2021-07,2021Q3,17,Monday,37,...,4,785.29,0.17,214.06,240.03,114.32,83,263,19922.49,High Calculated Value


In [167]:
# create product attributes table for sql analysis
product_attributes = df_final[[
    "customer_id",
    "product_id",
    "product_name",
    "product_category",
    "product_brand",
    "product_rating",
    "product_review_count",
    "product_stock",
    "product_return_rate",
    "product_size",
    "product_weight",
    "product_color",
    "product_material",
    "product_shelf_life"
]].copy()

product_attributes.head()

,customer_id,product_id,product_name,product_category,product_brand,product_rating,product_review_count,product_stock,product_return_rate,product_size,product_weight,product_color,product_material,product_shelf_life
0,1,1480,Product D,Electronics,Brand Y,2.5,560,48,0.40,Small,4.61,Red,Metal,250
1,2,1597,Product C,Groceries,Brand X,4.7,413,80,0.30,Medium,0.84,Blue,Metal,180
2,3,5142,Product B,Toys,Brand X,4.6,312,14,0.08,Medium,0.23,Green,Plastic,131
3,4,8447,Product A,Toys,Brand Z,1.1,110,69,0.09,Large,4.37,Blue,Wood,16
4,5,6025,Product C,Clothing,Brand X,3.8,172,25,0.39,Small,1.68,Red,Metal,57


In [168]:
# create promotion attributes table for sql analysis
promotion_attributes = df_final[[
     "customer_id",
    "promotion_id",
    "promotion_type",
    "promotion_start_date",
    "promotion_end_date",
    "promotion_effectiveness",
    "promotion_channel",
    "promotion_target_audience",
    "avg_discount_used",
    "total_discounts_received"
    
]].copy()

promotion_attributes.head()

,customer_id,promotion_id,promotion_type,promotion_start_date,promotion_end_date,promotion_effectiveness,promotion_channel,promotion_target_audience,avg_discount_used,total_discounts_received
0,1,271,20% Off,2021-07-14 14:28:42,2022-12-30 13:04:13,High,Online,New Customers,0.02,415.01
1,2,631,Flash Sale,2021-09-23 04:26:09,2022-09-13 03:16:26,Low,Social Media,New Customers,0.33,801.79
2,3,879,Flash Sale,2021-06-13 12:31:15,2022-03-13 00:53:35,Low,Online,New Customers,0.47,264.31
3,4,211,Buy One Get One Free,2021-05-23 05:42:48,2022-02-06 00:42:30,High,Social Media,Returning Customers,0.41,192.93
4,5,862,Flash Sale,2021-04-19 04:55:32,2022-12-04 13:07:09,Medium,Online,New Customers,0.22,497.26


In [169]:
#  reate store location attributes table for sql analysis

store_location_attributes = df_final[[
   
    "customer_id",
    "store_location",
    "store_city",
    "store_state",
    "store_zip_code",
    "distance_to_store",
    "holiday_season",
    "season",
    "weekend"

]].copy()

store_location_attributes.head()

,customer_id,store_location,store_city,store_state,store_zip_code,distance_to_store,holiday_season,season,weekend
0,1,Location A,City D,State Y,88500,33.21,No,Spring,Yes
1,2,Location C,City C,State X,30046,62.56,No,Summer,Yes
2,3,Location A,City A,State Y,26169,83.04,Yes,Winter,Yes
3,4,Location A,City B,State Z,22667,50.43,Yes,Winter,No
4,5,Location C,City C,State X,87843,36.55,Yes,Summer,Yes


In [170]:
import os

output_path = "../data/processed/sql_tables"

os.makedirs(output_path, exist_ok=True)

customers.to_csv(f"{output_path}/customers.csv", index=False)
customer_behavior.to_csv(f"{output_path}/customer_behavior.csv", index=False)
purchase_activity.to_csv(f"{output_path}/purchase_activity.csv", index=False)
product_attributes.to_csv(f"{output_path}/product_attributes.csv", index=False)
promotion_attributes.to_csv(f"{output_path}/promotion_attributes.csv", index=False)
store_location_attributes.to_csv(f"{output_path}/store_location_attributes.csv", index=False)


print("SQL tables have been created and saved to the processed data folder.")

SQL tables have been created and saved to the processed data folder.


## Additional SQL Practice Queries

After creating SQL-style analytical tables, I used DuckDB to practice interview-focused SQL queries. These queries are designed to demonstrate joins, aggregations, CASE statements, CTEs, window functions, ranking, and business-style reporting.

Because the dataset was found to be synthetic and evenly distributed, the goal of these queries is not to make strong business claims, but to practice SQL workflows and reporting logic.

In [171]:
import duckdb
conn = duckdb.connect()



In [172]:
#register tables

conn.register("customers", customers)
conn.register("customer_behavior", customer_behavior)
conn.register("purchase_activity", purchase_activity)
conn.register("product_attributes", product_attributes)
conn.register("promotion_attributes", promotion_attributes)
conn.register("store_location_attributes", store_location_attributes)


In [173]:
#Question which customer value segments have the highest average customer value and churn rate?

value_segment_summary = """
SELECT 
    p.calculated_value_segment,
    COUNT(c.customer_id) AS customer_count,
    ROUND(AVG(p.calculated_customer_value), 2) AS avg_calculated_value,
    ROUND(AVG(p.avg_transaction_value), 2) AS avg_transaction_value,
    ROUND(AVG(p.total_transactions), 2) AS avg_total_transactions,
    ROUND(AVG(c.churned_flag) * 100, 2) AS churn_rate
FROM customers c
JOIN purchase_activity p 
    ON c.customer_id = p.customer_id
GROUP BY p.calculated_value_segment
ORDER BY avg_calculated_value DESC;
"""

conn.execute(value_segment_summary).df()






,calculated_value_segment,customer_count,avg_calculated_value,avg_transaction_value,avg_total_transactions,churn_rate
0,High Calculated Value,333334,25878.04,364.58,72.48,49.94
1,Medium Calculated Value,333333,9930.31,246.84,49.06,50.00
2,Low Calculated Value,333333,2444.46,153.93,28.41,49.98


### SQL Query Interpretation: Churn by Customer Value Segment

This SQL query joined the purchase_activity table with the customers table using customer_id, then grouped customers by calculated_value_segment. The query calculated customer count, average customer value, average transaction value, average total transactions, and churn rate.

The results confirmed the earlier Python analysis. High Calculated Value customers had much higher average customer value, average transaction value, and total transactions. However, churn rates remained close to 50% across all customer value segments. This suggests that customer value does not meaningfully explain churn in this dataset.

In [174]:
# compare churn and customer value by lotalty program

query_loyalty_churn = """
select
c.loyalty_program,
count(*) as customer_count,
round(avg(p.calculated_customer_value),2) as avg_calculated_value,
round(avg(c.churned_flag)*100,2) as churn_rate,
round(median(p.calculated_customer_value),2) as median_calculated_value,
round(avg(p.avg_transaction_value),2) as avg_transaction_value,
round(avg(p.total_transactions),2) as avg_total_transactions,
from customers c
join purchase_activity p on c.customer_id = p.customer_id
group by c.loyalty_program
order by avg_calculated_value desc

"""

conn.execute(query_loyalty_churn).df()

,loyalty_program,customer_count,avg_calculated_value,churn_rate,median_calculated_value,avg_transaction_value,avg_total_transactions
0,Yes,499712,12760.46,49.96,9703.67,255.37,50.00
1,No,500288,12741.45,49.99,9624.98,254.86,49.97


### SQL Query Interpretation: Loyalty Program Performance


This query compared customer value and churn rate by loyalty program status. The results showed that loyalty program members and non-members had nearly identical churn rates, average transaction values, and average transaction counts.

Although loyalty members had a slightly higher average and median calculated customer value, the difference was very small. This suggests that loyalty program status does not meaningfully explain churn or customer value in this dataset.

In [175]:
#Income bracket analysis by customer value and churn

query_income_bracket = """
select
c.income_bracket,
count(*) as customer_count,
round(avg(p.calculated_customer_value),2) as avg_calculated_value,
round(avg(c.churned_flag)*100,2) as churn_rate,
round(median(p.calculated_customer_value),2) as median_calculated_value,
round(avg(p.avg_transaction_value),2) as avg_transaction_value,
round(avg(p.total_transactions),2) as avg_total_transactions,   
from customers c 
join purchase_activity p on c.customer_id = p.customer_id
group by c.income_bracket
order by avg_calculated_value desc
"""
conn.execute(query_income_bracket).df()

,income_bracket,customer_count,avg_calculated_value,churn_rate,median_calculated_value,avg_transaction_value,avg_total_transactions
0,Low,333021,12770.91,49.94,9709.50,255.32,50.03
1,Medium,333367,12754.37,50.04,9671.04,255.38,49.97
2,High,333612,12727.61,49.94,9613.69,254.65,49.97


### SQL Query Interpretation: Income Bracket Performance

This SQL query compared customer value and churn rate across income brackets. The results showed that Low, Medium, and High income customers had very similar average calculated customer values, transaction counts, average transaction values, and churn rates.

Although the Low income group had the highest average calculated customer value and slightly higher average total transactions, the differences were very small. This suggests that income bracket does not meaningfully explain customer value or churn in this dataset.

In [176]:
# Churn analysis by purchase frequency

query_purchase_frequency = """
select
b.purchase_frequency,
count(*) as customer_count,
round(median(p.calculated_customer_value),2) as median_calculated_value,
round(avg(p.calculated_customer_value),2) as avg_calculated_value,
round(avg(c.churned_flag)*100,2) as churn_rate,
round(avg(p.avg_transaction_value),2) as avg_transaction_value,
round(avg(p.total_transactions),2) as avg_total_transactions,

from customer_behavior b
join purchase_activity p
 on b.customer_id = p.customer_id
join customers c 
on b.customer_id = c.customer_id
group by b.purchase_frequency
order by avg_calculated_value desc
"""
conn.execute(query_purchase_frequency).df()

,purchase_frequency,customer_count,median_calculated_value,avg_calculated_value,churn_rate,avg_transaction_value,avg_total_transactions
0,Weekly,249768,9698.42,12784.67,50.08,255.31,50.08
1,Monthly,249932,9679.58,12759.03,49.80,254.91,50.03
2,Daily,249533,9640.80,12739.09,49.95,255.09,49.98
3,Yearly,250767,9636.66,12721.11,50.06,255.16,49.86


### SQL query Interpretation: Purchase Frequency Performance

This SQL query joined customer_behavior, purchase_activity, and customers to compare customer value and churn rates by purchase freuqency.  Weekly customers had the highest avgerage calculated customer value and average total transactions, but the difference between frequency groups was very small.

Churn rates stayed close to 50% across Daily, Weekly, Monthly, and Yearly customers.  Because of this, purchase_frequency does not appear to be a meaningful churn factor in this dataset.  


In [177]:
#  Question: can we bucket customers by recency and compare churn/value?

query_recency = """
Select
case
when b.days_since_last_purchase <= 30 then 'Recent'
when b.days_since_last_purchase <= 90 then 'At risk'
when b.days_since_last_purchase <= 180 then 'Dormant'
else 'Inactive'
end as recency_segment,
count(*) as customer_count,
round(median(p.calculated_customer_value),2) as median_calculated_value,
round(avg(p.calculated_customer_value),2) as avg_calculated_value,
round(avg(c.churned_flag)*100,2) as churn_rate,
round(avg(b.days_since_last_purchase),2) as avg_days_since_last_purchase,
round(avg(b.customer_support_calls),2) as avg_customer_support_calls,
from customer_behavior b
join purchase_activity p on b.customer_id = p.customer_id
join customers c on b.customer_id = c.customer_id
group by recency_segment
order by avg_calculated_value desc
"""
conn.execute(query_recency).df()


,recency_segment,customer_count,median_calculated_value,avg_calculated_value,churn_rate,avg_days_since_last_purchase,avg_customer_support_calls
0,Recent,84773,9679.46,12765.13,49.94,14.93,9.47
1,Inactive,504091,9671.62,12759.18,49.94,272.53,9.50
2,Dormant,246658,9696.66,12755.10,49.94,135.53,9.50
3,At risk,164478,9585.88,12712.19,50.13,60.50,9.50


### SQL Query Interpretation: Recency Segment Analysis

This query used a CASE statement to bucket customers into recency segments based on days_since_last_purchase. Most customers fell into the Inactive and Dormant segments, while Recent customers made up the smallest group.

Recent customers had the highest average calculated customer value, but the difference across recency segments was small. The At Risk segment had the highest churn rate at approximately 50.13%, but churn remained close to 50% across every recency group. Because the churn difference is very small, recency does not appear to meaningfully explain churn in this dataset.

This query is still useful for interview practice because it demonstrates how to create business segments using CASE statements, join multiple tables, aggregate metrics, and interpret results carefully.

In [178]:
# Lets calculate what percentage of customers fall into each recency segment
recency_query_by_segment = """
with recency_summary as(
    select
    CASE
    when b.days_since_last_purchase <= 30 then 'Recent'
    when b.days_since_last_purchase <= 90 then 'At risk'
    when b.days_since_last_purchase <= 180 then 'Dormant'
    else 'Inactive'
    end as recency_segment,
    count(*) as customer_count,
    round(avg(p.calculated_customer_value),2) as avg_calculated_value,
    round(median(p.calculated_customer_value),2) as median_calculated_value,
    round(avg(b.days_since_last_purchase),2) as avg_days_since_last_purchase,
    round(avg(b.customer_support_calls),2) as avg_customer_support_calls,
    round(avg(c.churned_flag)*100,2) as churn_rate

    from customer_behavior b
    join purchase_activity p on b.customer_id = p.customer_id
    join customers c on b.customer_id = c.customer_id
    group by
    CASE
    when b.days_since_last_purchase <= 30 then 'Recent'
    when b.days_since_last_purchase <= 90 then 'At risk'
    when b.days_since_last_purchase <= 180 then 'Dormant'
    else 'Inactive'
    end

)
select
recency_segment,
customer_count,
round(customer_count * 100.0/sum(customer_count) over(), 2) as percentage_of_customers, 
avg_calculated_value,
median_calculated_value,
avg_days_since_last_purchase,
avg_customer_support_calls,
churn_rate
from recency_summary
order by percentage_of_customers desc
"""

conn.execute(recency_query_by_segment).df()



,recency_segment,customer_count,percentage_of_customers,avg_calculated_value,median_calculated_value,avg_days_since_last_purchase,avg_customer_support_calls,churn_rate
0,Inactive,504091,50.41,12759.18,9671.62,272.53,9.50,49.94
1,Dormant,246658,24.67,12755.10,9696.66,135.53,9.50,49.94
2,At risk,164478,16.45,12712.19,9585.88,60.50,9.50,50.13
3,Recent,84773,8.48,12765.13,9679.46,14.93,9.47,49.94


### SQL Query Interpretation: Recency Segment CTE and Window Function

This query uses a Common Table Expression (CTE) to first group customers into recency segments based on days_since_last_purchase. Customers were categorized as Recent, At Risk, Dormant, or Inactive using a CASE statement.

The CTE calculates customer count, average calculated customer value, median calculated customer value, average days since last purchase, average support calls, and churn rate for each recency segment. The final SELECT statement then adds a percentage_of_customers field using a window function: SUM(customer_count) OVER().

The window function calculates the total customer count across all recency segments while keeping each segment as its own row. This allows the query to show what percentage of the full customer base belongs to each recency segment.

The results showed that most customers were classified as Inactive or Dormant, meaning a large portion of the customer base had not purchased recently. However, churn rates remained close to 50% across all recency groups, so recency does not appear to meaningfully explain churn in this dataset.

In [179]:
# Product Category Ranking by total calculated customer value using CTE and rank

category_ranking_query = """
WITH category_summary AS (
    SELECT
        pr.product_category,
        COUNT(*) AS customer_count,
        ROUND(SUM(p.calculated_customer_value), 2) AS total_calculated_value,
        ROUND(AVG(p.calculated_customer_value), 2) AS avg_calculated_value,
        ROUND(MEDIAN(p.calculated_customer_value), 2) AS median_calculated_value,
        ROUND(AVG(p.avg_transaction_value), 2) AS avg_transaction_value,
        ROUND(AVG(p.total_transactions), 2) AS avg_total_transactions
    FROM product_attributes pr
    JOIN purchase_activity p
        ON pr.customer_id = p.customer_id
    GROUP BY pr.product_category
)

SELECT
    product_category,
    customer_count,
    total_calculated_value,
    avg_calculated_value,
    median_calculated_value,
    avg_transaction_value,
    avg_total_transactions,
    RANK() OVER (ORDER BY total_calculated_value DESC) AS category_value_rank
FROM category_summary
ORDER BY category_value_rank;
"""

conn.execute(category_ranking_query).df()

,product_category,customer_count,total_calculated_value,avg_calculated_value,median_calculated_value,avg_transaction_value,avg_total_transactions,category_value_rank
0,Toys,200669,2.559170e+09,12753.19,9684.48,254.90,50.03,1
1,Groceries,200214,2.555058e+09,12761.63,9667.70,255.40,49.95,2
2,Clothing,199778,2.552960e+09,12778.98,9686.67,255.48,50.01,3
3,Furniture,199583,2.544757e+09,12750.37,9635.44,254.98,49.98,4
4,Electronics,199756,2.539005e+09,12710.53,9649.68,254.82,49.96,5


### SQL Query Interpretation: Product Category Ranking

This query used a CTE to summarize product categories by customer count, total calculated customer value, average calculated customer value, median calculated customer value, average transaction value, and average total transactions.

The final SELECT statement used the RANK() window function to rank product categories by total calculated customer value. This demonstrates how SQL can be used to create ranked business reports.

Because the dataset was found to be synthetic and evenly distributed, category differences should be interpreted carefully. The ranking is useful for SQL and reporting practice, but it should not be treated as a strong real-world product performance conclusion.

In [180]:
# Promotion type ranking by total caluclated customer value using CTE and rank\

promotion_ranking_query = """
WITH promotion_summary AS (
    SELECT
        promo.promotion_type,
        COUNT(*) AS customer_count,
        ROUND(SUM(p.calculated_customer_value), 2) AS total_calculated_value,
        ROUND(AVG(p.calculated_customer_value), 2) AS avg_calculated_value,
        ROUND(MEDIAN(p.calculated_customer_value), 2) AS median_calculated_value,
        ROUND(AVG(promo.avg_discount_used), 2) AS avg_discount_used,
        ROUND(AVG(promo.total_discounts_received), 2) AS avg_total_discounts_received,
        ROUND(AVG(c.churned_flag) * 100, 2) AS churn_rate
    FROM promotion_attributes promo
    JOIN purchase_activity p
        ON promo.customer_id = p.customer_id
    JOIN customers c
        ON promo.customer_id = c.customer_id
    GROUP BY promo.promotion_type
)

SELECT
    promotion_type,
    customer_count,
    total_calculated_value,
    avg_calculated_value,
    median_calculated_value,
    avg_discount_used,
    avg_total_discounts_received,
    churn_rate,
    RANK() OVER (ORDER BY total_calculated_value DESC) AS promotion_value_rank
FROM promotion_summary
ORDER BY promotion_value_rank;
"""

conn.execute(promotion_ranking_query).df()


,promotion_type,customer_count,total_calculated_value,avg_calculated_value,median_calculated_value,avg_discount_used,avg_total_discounts_received,churn_rate,promotion_value_rank
0,Buy One Get One Free,333520,4.260734e+09,12775.05,9698.86,0.25,500.24,49.96,1
1,20% Off,333712,4.251185e+09,12739.08,9651.25,0.25,499.71,49.99,2
2,Flash Sale,332768,4.239032e+09,12738.70,9645.29,0.25,499.08,49.97,3


### SQL Query Interpretation: Promotion Type Ranking

This query used a CTE to summarize customer value, discount behavior, and churn rate by promotion type. The final SELECT statement used RANK() to rank promotion types by total calculated customer value.

The query is useful for practicing SQL reporting logic, including joins, aggregations, CTEs, and window functions. However, because promotion fields were found to be synthetic and evenly distributed, the results should be treated as descriptive SQL practice rather than strong evidence of promotion effectiveness.

In [181]:
# Website engagement segment using case statements

website_engagement_query = """
select
CASE 
when website_visits >= 75 then 'High Website Engagement'
when website_visits >= 25 then 'Medium Website Engagement'
else 'Low Website Engagement'
END as website_engagement_segment,
count(*) as customer_count,
round(avg(p.calculated_customer_value),2) as avg_calculated_value,
round(median(p.calculated_customer_value),2) as median_calculated_value,
round(avg(b.website_visits),2) as avg_website_visits,
round(avg(b.customer_support_calls),2) as avg_customer_support_calls,
round(avg(c.churned_flag)*100,2) as churn_rate
from customer_behavior b
join purchase_activity p 
on b.customer_id = p.customer_id
join customers c 
on b.customer_id = c.customer_id
group by 
CASE
when website_visits >= 75 then 'High Website Engagement'
when website_visits >= 25 then 'Medium Website Engagement'
else 'Low Website Engagement'
END
order by avg_calculated_value desc
"""

conn.execute(website_engagement_query).df()


,website_engagement_segment,customer_count,avg_calculated_value,median_calculated_value,avg_website_visits,avg_customer_support_calls,churn_rate
0,High Website Engagement,250148,12780.02,9708.20,87.00,9.53,49.95
1,Low Website Engagement,249923,12757.08,9666.57,11.99,9.50,50.05
2,Medium Website Engagement,499929,12733.34,9643.90,49.51,9.48,49.95


### SQL Query Interpretation: Website Engagement Segments

This query used a CASE statement to group customers into Low, Medium, and High Website Engagement segments based on website visit count. It then compared customer count, calculated customer value, website visits, support calls, and churn rate across each segment.

This query demonstrates how SQL can transform raw numeric fields into business-friendly reporting segments. In this dataset, engagement segments should be interpreted cautiously because earlier analysis showed that engagement variables had little to no meaningful relationship with churn.

In [182]:
# Above average customer value segments with multiple CTE'

above_avg_customer_value = """
WITH segment_summary AS (
    SELECT
        p.calculated_value_segment,
        COUNT(*) AS customer_count,
        ROUND(AVG(p.calculated_customer_value), 2) AS avg_calculated_value,
        ROUND(MEDIAN(p.calculated_customer_value), 2) AS median_calculated_value,
        ROUND(AVG(c.churned_flag) * 100, 2) AS churn_rate
    FROM purchase_activity p
    JOIN customers c
        ON p.customer_id = c.customer_id
    GROUP BY p.calculated_value_segment
),

overall_average AS (
    SELECT
        ROUND(AVG(calculated_customer_value), 2) AS overall_avg_value
    FROM purchase_activity
)

SELECT
    s.calculated_value_segment,
    s.customer_count,
    s.avg_calculated_value,
    s.median_calculated_value,
    o.overall_avg_value,
    s.churn_rate
FROM segment_summary s
CROSS JOIN overall_average o
WHERE s.avg_calculated_value > o.overall_avg_value
ORDER BY s.avg_calculated_value DESC;
"""

conn.execute(above_avg_customer_value).df()



,calculated_value_segment,customer_count,avg_calculated_value,median_calculated_value,overall_avg_value,churn_rate
0,High Calculated Value,333334,25878.04,24246.11,12750.95,49.94


### SQL Query Interpretation: Above-Average Customer Value Segments

This query used two CTEs. The first CTE summarized customer value and churn rate by calculated value segment. The second CTE calculated the overall average customer value across the full dataset.

The final SELECT used a CROSS JOIN to compare each segment's average customer value to the overall dataset average. This returned only the segments with above-average customer value.

This query is useful for interview practice because it demonstrates multiple CTEs, aggregate comparison logic, CROSS JOIN usage, filtering summarized results, and business-style reporting.

## Export Summary Tables for Excel and Google Sheets Practice

After completing SQL practice queries, I exported several summarized tables to CSV files. These exports can be used in Excel or Google Sheets to practice pivot tables, charts, conditional formatting, KPI cards, and dashboard-style reporting.

The exported tables are intended for reporting practice, not for making strong business recommendations, because the dataset was found to be highly synthetic and evenly distributed.

In [183]:
import os

excel_export_path = "../Exports"
os.makedirs(excel_export_path, exist_ok=True)

print("Export path:", os.path.abspath(excel_export_path))

Export path: c:\Users\giova\Desktop\Retail Sales and Customer behavior intelligence analysis\Exports


In [185]:
value_summary_export = conn.execute(value_segment_summary).df()
value_summary_export.to_csv(f"{excel_export_path}/customer_value_segment_summary.csv", index=False)

loyalty_churn_export = conn.execute(query_loyalty_churn).df()
loyalty_churn_export.to_csv(f"{excel_export_path}/loyalty_program_churn_summary.csv", index=False)

income_churn_export = conn.execute(query_income_bracket).df()
income_churn_export.to_csv(f"{excel_export_path}/income_bracket_churn_summary.csv", index=False)

purchase_frequency_export = conn.execute(query_purchase_frequency).df()
purchase_frequency_export.to_csv(f"{excel_export_path}/purchase_frequency_churn_summary.csv", index=False)

recency_export = conn.execute(query_recency).df()
recency_export.to_csv(f"{excel_export_path}/recency_segment_summary.csv", index=False)

category_ranking_export = conn.execute(category_ranking_query).df()
category_ranking_export.to_csv(f"{excel_export_path}/product_category_ranking.csv", index=False)

promotion_ranking_export = conn.execute(promotion_ranking_query).df()
promotion_ranking_export.to_csv(f"{excel_export_path}/promotion_type_ranking.csv", index=False)

website_engagement_export = conn.execute(website_engagement_query).df()
website_engagement_export.to_csv(f"{excel_export_path}/website_engagement_summary.csv", index=False)

above_avg_value_export = conn.execute(above_avg_customer_value).df()
above_avg_value_export.to_csv(f"{excel_export_path}/above_average_customer_value_segments.csv", index=False)



In [184]:
# 4. Verify export
os.listdir(excel_export_path)

[]